# Lab 10 — Web Scraping a Static Public Page
**Data Acquisition Track** · Intermediate · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Parse an HTML table with html.parser (BeautifulSoup optional)
2. Normalise rows to dicts and handle missing cells
3. Export scraped.csv with a row-count assertion
4. Practice polite scraping rules and robots.txt awareness

## Datasets (this folder)
- `fixture.html`

## How to run on Google Colab
1. Open this notebook from the Kongu AI Academy lab page (**Open in Colab**). Colab saves a copy to your Google Drive.
2. Run **Cell 0** first: it downloads and unzips the dataset.
3. Write your code in the exercise cells and run them.
4. Run the last cell, `kaa_check()`, and compare your answers with the expected outputs in the exercises.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`fixture.html`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — Kongu AI Academy: download the dataset (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-10-web-scraping-static-page"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/abhisek8113/kongu-ai-academy/main/public/course-labs/course-01-foundations-python-math-data/lab-10-web-scraping-static-page/bundle/dataset.zip"
NEED = ["fixture.html"]  # unzipped files used by the code below
EXERCISES = ["units", "mean_price"]  # functions you write in the exercise cells


def kaa_check():
    """Self-check: runs each exercise function and prints what it returns."""
    for name in EXERCISES:
        fn = globals().get(name)
        if not callable(fn):
            print(f"✗ {name}: not written yet")
            continue
        try:
            print(f"✓ {name}() ->", fn())
        except TypeError:
            print(f"✓ {name} is defined (takes arguments; test it with the example in its exercise)")
        except Exception as exc:
            print(f"✗ {name} raised {type(exc).__name__}: {exc}")


def _have_files():
    return all(os.path.exists(f) for f in NEED)


if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", DATASET_ZIP_URL, "-O", "dataset.zip"], check=True)
    else:
        urllib.request.urlretrieve(DATASET_ZIP_URL, "dataset.zip")
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))

## Data Acquisition Track: HTML Tables → CSV (Offline Fixture First)

> **Scenario:** Extract a product table from HTML with `urllib` + `html.parser` (or BeautifulSoup if available), handle missing cells, and save `scraped.csv`. A local `fixture.html` ships with this lab so **everything runs without network**. An optional live target (`quotes.toscrape.com`) is included for when you do have egress.
>
> **You will learn:** HTML structure, table parsing, polite scraping (User-Agent, delay), robots.txt awareness, anti-patterns warning.
> **Time:** ~45 minutes. **Level:** Intermediate. **Needs:** Python 3.8+ (BeautifulSoup optional). **Env:** 🟢 Colab only.

Every scraping job is the same three-step pipeline: **fetch → parse → validate**. This lab front-loads the *parse* and *validate* steps against a local `fixture.html`, so you can debug table logic without flaky network behaviour. Only at the very end do we touch the *fetch* step, behind a flag. Treat that order as the house style for any future data-acquisition work: prove your parsing offline first, then go live.

### Scraping mental map

The browser renders a DOM tree; Python sees a stream of tags and text. Mapping between the two is the whole game — once you can point at “the `<tr>` rows inside `table#products`”, the rest is bookkeeping:

| HTML piece | Python view | Extract how |
|---|---|---|
| `<table>` | container | `HTMLParser` handle_starttag |
| `<tr>` | row | accumulate cells |
| `<td>` | cell text | `handle_data` between tags |
| empty `<td>` | missing value | `""` → flag / impute |
| CSS selector | BeautifulSoup | `soup.select("table#products tr")` |

> **Politeness rules:** identify yourself with a `User-Agent`, cache results, don’t hammer servers, and respect `robots.txt` + ToS. Scraping behind logins or against explicit disallow paths is not okay.

> **Pitfall:** robots/ToS are not suggestions. Before any live fetch, read the site’s `robots.txt` and terms of service; if scraping is disallowed, or the data sits behind a login, use a sanctioned API or request export instead. A training lab is not a licence to crawl production systems.

---

### 1. Load local fixture (offline path — always works)

Why this comes first: a network failure and a parser bug look identical when you only run live code (“zero rows again…”). Reading `fixture.html` from disk removes the network variable, so when row counts are wrong you know the bug is in your parser, not in DNS, timeouts, or a server-side change.


In [ ]:
import csv, os, re, time
from html.parser import HTMLParser

FIXTURE = "fixture.html"
LIVE_URL = "https://quotes.toscrape.com/"  # optional, public practice site

def read_local(path=FIXTURE):
    with open(path, encoding="utf-8") as f:
        return f.read()

html = read_local()
print("fixture bytes:", len(html))
assert "<table" in html and "Widget A" in html


**What to notice:** the printed byte count is non-zero, and the `assert` passes — the string `"<table"` proves a table element exists and `"Widget A"` proves we loaded the real fixture rather than an empty or truncated file. That two-part check (structure marker + known data value) is the cheapest possible smoke test; run it on every scrape.

If the fixture is missing, the notebook writes a minimal one:


In [ ]:
if not os.path.exists(FIXTURE):
    open(FIXTURE, "w", encoding="utf-8").write(
        "<table id='products'><tr><th>ID</th><th>Name</th><th>Price</th><th>Stock</th></tr>"
        "<tr><td>1</td><td>Widget A</td><td>9.99</td><td>120</td></tr></table>"
    )


**What to notice:** the generated fallback has one header row (`ID, Name, Price, Stock`) and a single data row (`Widget A`), so downstream code still exercises the same parse path — it just yields fewer rows than the full five-row fixture.

---

### 2. Stdlib parser: table rows → list of lists

Why use the standard library here: `html.parser` ships with Python, so this cell works in any environment with zero installs. The parser is a small state machine — `in_tr` tracks “am I inside a row?”, `in_td` tracks “am I collecting cell text?”, and `handle_data` appends characters only while a cell is open. Understanding this machine makes BeautifulSoup’s convenience methods obvious rather than magical.


In [ ]:
class TableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_td = False
        self.in_tr = False
        self.cell = ""
        self.row = []
        self.rows = []
        self.in_header = False

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self.in_tr = True
            self.row = []
        elif tag in ("td", "th"):
            self.in_td = True
            self.cell = ""
        if tag == "th":
            self.in_header = True

    def handle_endtag(self, tag):
        if tag in ("td", "th"):
            self.in_td = False
            self.row.append(self.cell.strip())
        elif tag == "tr":
            self.in_tr = False
            if self.row:
                self.rows.append(self.row)
            self.in_header = False

    def handle_data(self, data):
        if self.in_td:
            self.cell += data

def parse_table(html_text):
    p = TableParser()
    p.feed(html_text)
    if not p.rows:
        return [], []
    header, *body = p.rows
    return header, body

header, body = parse_table(html)
print("header:", header)
# ['ID', 'Name', 'Price', 'Stock']
print("n data rows:", len(body))
print("first row:", body[0])


**What to notice:** the header list has exactly four names (`['ID', 'Name', 'Price', 'Stock']`), the body count is **5** data rows — header row excluded — and the first row is `['1', 'Widget A', '9.99', '120']` with every value already a stripped string. Cells come out as text on purpose: `9.99` is a string here and only becomes a `float` in the next section, where you control the coercion and its failure modes.

---

### 3. Rows → dicts; handle missing cells

Fixture intentionally has a **blank Price** on Gadget C. Real tables are full of blanks, `N/A`, and stray whitespace; a parser that assumes every cell is populated will either crash or — worse — silently write garbage. The pattern below pads short rows to the header width, then converts each numeric field with an explicit fallback to `None`.


In [ ]:
def rows_to_dicts(header, body):
    out = []
    for r in body:
        # pad short rows
        r = list(r) + [""] * (len(header) - len(r))
        rec = dict(zip(header, r))
        # normalise numerics with fallbacks
        rec["Price"] = float(rec["Price"]) if rec["Price"] not in ("", None) else None
        rec["Stock"] = int(rec["Stock"]) if str(rec["Stock"]).strip().isdigit() else None
        out.append(rec)
    return out

records = rows_to_dicts(header, body)
for rec in records:
    print(rec)
# ...

missing_price = [r for r in records if r["Price"] is None]
print("missing prices:", len(missing_price))


**What to notice:** `Widget A` normalises cleanly to `Price: 9.99` (a real `float`) and `Stock: 120` (a real `int`), while **Gadget C** carries `Price: None` — exactly **1** missing price across the table. `None` is the right sentinel: it is falsy but distinct from `0`, so a genuinely free item (price `0.0`) never gets confused with an unknown price.

> Always assert row counts and required keys after parsing — silent empty scrapes are the #1 bug.

> **Pitfall:** never impute a missing price as `0`. Zero is a valid price; `None` (or `NaN`) keeps “unknown” separate from “free”, and downstream averages skip `None` instead of being dragged toward zero.

---

### 4. Write `scraped.csv` + row-count assert

Why a write-then-reopen check: `csv.DictWriter` can succeed while still dropping or mangling rows (wrong fieldnames, a `None` that stringifies oddly). Reading the file back and counting lines turns “I think it wrote” into “I proved it wrote”, and the `assert` fails loudly in CI or a graded notebook instead of shipping an empty artefact.


In [ ]:
out_path = "scraped.csv"
fields = ["ID", "Name", "Price", "Stock"]
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in records:
        w.writerow({
            "ID": r["ID"],
            "Name": r["Name"],
            "Price": "" if r["Price"] is None else f"{r['Price']:.2f}",
            "Stock": "" if r["Stock"] is None else r["Stock"],
        })

with open(out_path, encoding="utf-8") as f:
    n = sum(1 for _ in f)
assert n == 1 + len(records), f"expected {1+len(records)} lines, got {n}"
print(f"wrote {out_path}: {len(records)} data rows, {n} lines total")


**What to notice:** **5 data rows** plus one header line gives **6 lines total**, and the `assert` encodes exactly that identity (`n == 1 + len(records)`). Missing prices are written as empty CSV fields (`""`), not the string `"None"` — so a re-reader of `scraped.csv` sees a blank cell, not literal text that would poison a later `float()` conversion.

---

### 5. Optional: BeautifulSoup (if installed)

Why bother if the stdlib parser works: BeautifulSoup gives you CSS selectors (`table#products tbody tr`), tolerant error recovery, and `get_text(strip=True)` helpers that scale to messier pages. Keep it optional in this lab — the `try/except ImportError` guarantees the notebook still runs where `bs4` is absent, and the printed row count lets you confirm both implementations agree.


In [ ]:
try:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    trs = soup.select("table#products tbody tr")
    print("bs4 rows:", len(trs))
    bs_rows = []
    for tr in trs:
        tds = [td.get_text(strip=True) for td in tr.find_all("td")]
        bs_rows.append(tds)
    print(bs_rows[0])
except ImportError:
    print("BeautifulSoup not installed — stdlib parser above is sufficient")


**What to notice:** BeautifulSoup reports **5** rows — the same count as the stdlib parser — and its first row is the identical `['1', 'Widget A', '9.99', '120']`. When two independent parsers agree on both count and content, you have real cross-validation; a mismatch would mean one of them is mishandling `thead`/`tbody` structure or nested tags.

---

### 6. Optional live fetch (network only)

Why this is last and gated: live pages change, rate-limit, and introduce robots constraints that have nothing to do with your parsing logic. The `fetch_live` helper already does the polite things — a 1-second `delay`, an explicit educational `User-Agent`, and a timeout — but lab verification deliberately runs offline (the print at the bottom) so grading never depends on third-party uptime.


In [ ]:
def fetch_live(url, delay=1.0, timeout=10):
    import urllib.request
    time.sleep(delay)  # be polite
    req = urllib.request.Request(url, headers={"User-Agent": "ailab-lab10/1.0 (educational)"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")

# Live scrape is optional; lab verification uses fixture.html only.
# if os.environ.get("AILAB_ALLOW_NET"):
#     page = fetch_live(LIVE_URL)
#     print("live bytes:", len(page))
print("offline mode: skipping live fetch (set AILAB_ALLOW_NET=1 to enable)")


**What to notice:** the cell prints `offline mode: skipping live fetch (set AILAB_ALLOW_NET=1 to enable)` — no socket was opened, so the run is reproducible. When you do opt in, put `time.sleep(delay)` *before* each request, identify yourself honestly in the `User-Agent`, and cache the response to disk so re-runs don’t hammer the server.

> Check `https://quotes.toscrape.com/robots.txt` before scraping any live site. Practice domains exist for a reason.

> **Pitfall:** obey robots.txt and the site’s ToS on every live target — including “practice” domains. If the file disallows your path, stop; if the terms forbid automated collection, use the official API or a provided dataset. Politeness (honest User-Agent, delays, caching) is the minimum, not the ceiling.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

### Exercise 1 — Parse table rows to dicts
Using `TableParser` + `rows_to_dicts`, print the number of rows and the full list of `Name` values.


**Follow-up:** Add the stock count of every product. Return the total number of units in stock, not the average price.

<details>
<summary>Hint</summary>

`header, body = parse_table(html)` then `rows_to_dicts(header, body)`; collect `r["Name"]`.
</details>

### Exercise 2 — Handle missing cells
Which product has an empty `Price`? How many missing prices? What Python value do you store?


**Follow-up:** Ignore products that have no price. Return the mean of the prices that are present. Do not treat a missing price as zero.

<details>
<summary>Hint</summary>

`[r for r in records if r["Price"] is None]`.
</details>

### Exercise 3 — Write CSV + row-count assert
Write `scraped.csv` and assert `lines == 1 + len(records)`. Print the assertion message.


**Follow-up:** Prove the CSV header.

<details>
<summary>Hint</summary>

`sum(1 for _ in open(path))` includes the header.
</details>

---


In [ ]:
def units():
    """Add the stock count of every product. Return the total number of units in stock, not the average price."""
    pass


In [ ]:
def mean_price():
    """Ignore products that have no price. Return the mean of the prices that are present. Do not treat a missing price as zero."""
    pass


In [ ]:
kaa_check()  # prints your answers so you can compare them with the expected outputs


**What to notice:** the stock total asserts to exactly **472** units, the mean priced product lands at about **12.685** (Gadget C’s `None` correctly excluded from both sum and divisor), and the CSV header line is the exact string `ID,Name,Price,Stock`. If any of the three differs, the bug is upstream: re-check padding in `rows_to_dicts`, `None` handling, or the `fieldnames` order in Section 4.

### What to learn next
- CSS selectors with BeautifulSoup: `soup.select_one("td.price")`.
- APIs behind the page (view Network tab) often beat HTML scraping.
- `scrapy` / `playwright` for JS-heavy sites — and legal review first.
- Cheat sheet: fixture-first → parse → normalise missings → assert counts → export → only then go live.

*Files in this folder: `fixture.html` (offline table) · output `scraped.csv`.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
